[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_27_Deep_Belief_Network_Representation_Learning.ipynb)


# Experiment 27: Deep Belief Network Representation Learning

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 27 – Deep Belief Network Representation Learning

In [ ]:
# Greedy stacked RBM pretraining on MNIST, then supervised fine-tuning
import tensorflow as tf
from sklearn.datasets import load_digits
digits=load_digits(); X=digits.data.astype("float32")/16; y=digits.target
# Use digits (8x8) as a lightweight MNIST substitute
def train_rbm_layer(data,n_hidden,epochs=5):
    n_visible=data.shape[1]; w=np.random.normal(0,.01,(n_visible,n_hidden)); vb=np.zeros(n_visible); hb=np.zeros(n_hidden)
    for _ in range(epochs):
        hp=sig(data@w+hb); hs=sample_prob(hp); vp=sig(hs@w.T+vb); vs=sample_prob(vp); hp1=sig(vs@w+hb)
        w+=.05*(data.T@hp-vs.T@hp1)/len(data); vb+=.05*np.mean(data-vs,0); hb+=.05*np.mean(hp-hp1,0)
    return w,hb
w1,b1=train_rbm_layer(X,128); h1=sig(X@w1+b1)
w2,b2=train_rbm_layer(h1,64)
plt.imshow(w1.T[:25].reshape(-1,8,8)[0],cmap="gray"); plt.title("Example first-layer learned filter"); plt.show()
# Transfer pretrained weights into a classifier
model=tf.keras.Sequential([tf.keras.layers.Input((64,)),tf.keras.layers.Dense(128,activation="sigmoid"),tf.keras.layers.Dense(64,activation="sigmoid"),tf.keras.layers.Dense(10,activation="softmax")])
model.layers[0].set_weights([w1,b1]); model.layers[1].set_weights([w2,b2])
model.compile("adam","sparse_categorical_crossentropy",metrics=["accuracy"])
xtr,xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
model.fit(xtr,ytr,epochs=15,verbose=0); print("Pretrained test accuracy:",model.evaluate(xte,yte,verbose=0)[1])